# Tutorial 5 — CNN classification on CIFAR 10

Build a CNN, calculate output sizes and parameter counts, compare filters and Dense layers, and train a regularized variant.

**How to run:** select a Python kernel with the repository requirements, then Run All. QUICK mode keeps experiments short. For the larger run set `TUTORIAL_FULL=1` before starting the kernel. A quick run demonstrates the method; it is not a final accuracy benchmark.

**Execution status:** Complete implementation; end-to-end training/inference has not been verified in this repository run because the required large downloads did not finish. See `VALIDATION.md` for the separate offline model checks. Run All in Colab to generate your own results.

In [ ]:
# Google Colab: fetch the support files and install missing dependencies.
# Local Jupyter: follow README setup once; this cell makes no changes.
import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    repo = Path("/content/dl-course-tutorials")
    if not repo.exists():
        subprocess.check_call(["git", "clone", "https://github.com/naveed952/Deep_learning-Tutorials_py.ipynb.git", str(repo)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo/"requirements.txt")])
    import os
    os.chdir(repo)


## Step 1 — Locate the project and set a reproducible seed
A fixed seed makes comparisons easier to reproduce. Each notebook can run independently.

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path.cwd() if (Path.cwd()/"tutorial_support").exists() else Path.cwd().parent
assert (ROOT/"tutorial_support").exists(), "Run from the repository or notebooks folder."
sys.path.insert(0, str(ROOT))
from tutorial_support.common import *
seed()
print("Mode:", "QUICK: small real-data run" if QUICK else "FULL: larger training run")

## Step 2 — Prepare real CIFAR 10 images
Use a validation split from training data. Unlike the handout demonstration, we do not repeatedly select models on the test set.

In [ ]:
keras=keras_setup()
from sklearn.model_selection import train_test_split
import pandas as pd,matplotlib.pyplot as plt
(x,y),(xt,yt)=cifar10_arrays();y=y.ravel();yt=yt.ravel()
if QUICK:x,y=x[:3000],y[:3000];xt,yt=xt[:500],yt[:500]
xtr,xv,ytr,yv=train_test_split(x.astype('float32')/255,y,test_size=.2,stratify=y,random_state=SEED)
xt=xt.astype('float32')/255
classes=['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
fig,axes=plt.subplots(1,8,figsize=(12,2))
for i,ax in enumerate(axes):ax.imshow(xtr[i]);ax.set_title(classes[ytr[i]]);ax.axis('off')
plt.show()

## Step 3 — Define and inspect the CNN
A 3×3 RGB convolution with 32 filters has (3×3×3+1)×32 = 896 parameters. Pooling reduces spatial dimensions without learned weights. Sparse cross-entropy accepts integer labels.

In [ ]:
def build_cnn(filters=(32,64),kernel=3,dense=64,regularized=False):
    keras.utils.set_random_seed(SEED)
    layers=[keras.layers.Input((32,32,3))]
    if regularized:layers += [keras.layers.RandomFlip('horizontal'),keras.layers.RandomRotation(.05)]
    for f in filters:
        layers += [keras.layers.Conv2D(f,kernel,padding='same',activation='relu',kernel_regularizer=keras.regularizers.l2(1e-4) if regularized else None),keras.layers.MaxPool2D()]
    layers += [keras.layers.Flatten(),keras.layers.Dense(dense,activation='relu')]
    if regularized:layers += [keras.layers.Dropout(.3)]
    layers += [keras.layers.Dense(10,activation='softmax')]
    model=keras.Sequential(layers);model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return model
base=build_cnn();base.summary();assert base.layers[0].count_params()==896

## Step 4 — Complete the architecture and improvement experiments
The regularized variant adds augmentation, L2 and dropout. Early stopping uses validation loss. More filters increase parameters; they do not guarantee better validation accuracy.

In [ ]:
configs=[('baseline',{}),('kernel5',{'kernel':5}),('more_filters',{'filters':(32,64,128)}),('larger_dense',{'dense':128}),('regularized',{'regularized':True})]
histories={};models={};rows=[]
for name,kw in configs:
    model=build_cnn(**kw)
    h=model.fit(xtr,ytr,validation_data=(xv,yv),epochs=3 if QUICK else 20,batch_size=64,verbose=0,callbacks=[keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True)])
    models[name]=model;histories[name]=h;vl,va=model.evaluate(xv,yv,verbose=0)
    rows.append({'model':name,'params':model.count_params(),'val_loss':vl,'val_accuracy':va})
results=pd.DataFrame(rows).sort_values('val_loss');display(results);curves(histories);curves(histories,'accuracy')
chosen=results.iloc[0]['model'];test=models[chosen].evaluate(xt,yt,verbose=0);print('Selected:',chosen,'test:',test)
save_result('05_cnn',{'comparison':rows,'selected':chosen,'test':test})

## Step 5 — A convolution and its gradient
The same filter is reused at every location. Its gradient sums contributions from every receptive field. This small example checks the derivative numerically.

In [ ]:
import torch
patch=torch.arange(1.,10.).reshape(1,1,3,3)
kernel=torch.ones((1,1,2,2),requires_grad=True)
output=torch.nn.functional.conv2d(patch,kernel)
loss=output.sum();loss.backward();grad=kernel.grad
print('Feature map:',output.detach().numpy().squeeze());print('Gradient:',grad.numpy().squeeze())
assert np.allclose(grad.numpy().squeeze(),[[12,16],[24,28]])
with torch.no_grad():kernel-=.01*grad
print('Updated filter:',kernel.detach().numpy().squeeze())

## Explain it in class
Describe the input, target, model, loss, update rule and evaluation split. Use the displayed results from your own run. Changing the dataset, seed or training budget can change the result.

**Course source:** the matching tutorial PDF supplied in the Deep Learning course. Original explanatory implementation; the course PDFs are not redistributed.

## Task answers and revision check

**Before reporting performance:** run every comparison cell and retain its curves/table. Full CIFAR-10 training has not been verified in the saved repository run; synthetic gradient checks do not establish classification accuracy.

**Filter-size question:** a 5×5 kernel has more weights than a 3×3 kernel at the same channel counts. SAME padding preserves height/width for stride 1, but not the parameter count. More convolution blocks also reduce spatial dimensions through pooling.

**Numerical answer:** a 32×32×3 image passed through 32 filters of size 3×3, stride 1, SAME padding becomes 32×32×32. The convolution has 896 parameters. A following 2×2 stride-2 valid max pool produces 16×16×32 with no learned parameters.

**What should the comparison conclude?** Report which model had the best validation result, its parameter count and the selected model's test result. Do not state that adding filters or dropout improved accuracy unless the measured values show it. Regularization penalties affect total loss; also compare validation accuracy.

**Gradient answer:** the same filter is reused over the image, so its gradient sums contributions from all output positions. Max pooling routes gradient to the selected maximum in each window.
